# PyTorch Quickstart

This notebook adapts the official PyTorch quickstart tutorial for this Quarto Book.

Source: [PyTorch Tutorials - Quickstart](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)

Learning goals:

- Load a real dataset with `torchvision.datasets`.
- Wrap a `Dataset` with `torch.utils.data.DataLoader`.
- Define a neural network with `nn.Module`.
- Train and evaluate the model with explicit loops.
- Save, load, and use model weights for prediction.

## Working with data

PyTorch has two core primitives for data work: `torch.utils.data.Dataset` and `torch.utils.data.DataLoader`. `Dataset` stores samples and labels; `DataLoader` wraps a dataset as an iterable with batching, sampling, shuffling, and multiprocessing support.


In [6]:
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

from src.utils import find_project_root

In [11]:
PROJECT_ROOT = find_project_root(Path.cwd())
_ = torch.manual_seed(42)

In [8]:
%matplotlib inline

PyTorch offers domain-specific libraries such as
[TorchText](https://pytorch.org/text/stable/index.html),
[TorchVision](https://pytorch.org/vision/stable/index.html), and
[TorchAudio](https://pytorch.org/audio/stable/index.html), all of which
include datasets. For this tutorial, we will be using a TorchVision
dataset.

The `torchvision.datasets` module contains `Dataset` objects for many
real-world vision data like CIFAR, COCO ([full list
here](https://pytorch.org/vision/stable/datasets.html)). In this
tutorial, we use the FashionMNIST dataset. Every TorchVision `Dataset`
includes two arguments: `transform` and `target_transform` to modify the
samples and labels respectively.


In [9]:
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "fashion-mnist"

# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=True,
    download=False,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=False,
    download=False,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

We pass the `Dataset` as an argument to `DataLoader`. This wraps an
iterable over our dataset, and supports automatic batching, sampling,
shuffling and multiprocess data loading. Here we define a batch size of
64, i.e. each element in the dataloader iterable will return a batch of
64 features and labels.


In [5]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


> What to notice: images arrive as batches shaped `[N, C, H, W]`: batch size, channel count, height, and width. FashionMNIST has one grayscale channel and 28 x 28 images.

Read more: [Datasets and DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html).

## Creating Models

To define a neural network in PyTorch, create a class that inherits from [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). Define layers in `__init__`, then describe the data flow in `forward`.

For acceleration, move the model and tensors to an available accelerator such as CUDA, MPS, MTIA, or XPU. If no accelerator is available, use CPU.

In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)
print(model)

Using mps device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


> What to notice: `forward()` defines the model computation, but user code normally calls `model(x)`. `nn.Module.__call__` handles hooks and then dispatches into `forward()`.

Read more: [Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html).

## Optimizing the Model Parameters

To train a model, define a [loss function](https://pytorch.org/docs/stable/nn.html#loss-functions) and an [optimizer](https://pytorch.org/docs/stable/optim.html). The loss measures prediction error; the optimizer updates model parameters from gradients.

In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In a single training loop, the model makes predictions on the training
dataset (fed to it in batches), and backpropagates the prediction error
to adjust the model\'s parameters.


In [8]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

We also check the model\'s performance against the test dataset to
ensure it is learning.


In [9]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

The training process runs for several iterations, called *epochs*. Each epoch gives the model another pass over the training data.

This example uses five epochs. On a laptop CPU or Apple Silicon MPS, FashionMNIST should stay reasonably quick.

In [10]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.301212  [   64/60000]


loss: 2.295208  [ 6464/60000]


loss: 2.283874  [12864/60000]


loss: 2.262271  [19264/60000]


loss: 2.255660  [25664/60000]


...

loss: 1.143658  [38464/60000]


loss: 1.138430  [44864/60000]


loss: 1.147276  [51264/60000]


loss: 1.111481  [57664/60000]


Test Error: 
 Accuracy: 64.3%, Avg loss: 1.087230 

Done!


> What to notice: the loss should generally decrease and the accuracy should generally improve. Small fluctuations are normal because training uses shuffled mini-batches.

Read more: [Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).

## Saving Models

A common way to save a model is to serialize the internal state dictionary, which contains the learned model parameters. Keep generated model artifacts under `artifacts/` so they do not clutter the source tree.

In [11]:
MODEL_PATH = PROJECT_ROOT / "artifacts" / "basic" / "quickstart" / "model.pth"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), MODEL_PATH)
print(f"Saved PyTorch model state to {MODEL_PATH.relative_to(PROJECT_ROOT)}")

Saved PyTorch model state to artifacts/basic/quickstart/model.pth


## Loading Models

To load a model, recreate the model structure first, then load the saved state dictionary into it.

In [12]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))

<All keys matched successfully>

This model can now be used to make predictions.


In [13]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


> What to notice: inference uses the same model class plus loaded weights. The prediction can still be wrong; this quickstart is about the workflow, not maximizing FashionMNIST accuracy.

Read more: [Save and Load the Model](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html).